# NLI observacional no golden set
Use uma GPU **L4** (ou T4, se L4 não estiver disponível). Este notebook não treina nem altera o serviço: compara o NLI com o gold e com o relatório-base do LLM.

In [ ]:
!pip -q install 'transformers>=4.45,<5' sentencepiece safetensors
from google.colab import files
uploaded = files.upload()  # envie golden_claims.jsonl e claims.json


In [ ]:
import json, time
from pathlib import Path
import torch
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
REVISION = 'b5113eb'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')
assert DEVICE.type == 'cuda', 'Ative uma GPU no menu Ambiente de execução > Alterar tipo de ambiente.'

items = [json.loads(line) for line in Path('golden_claims.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
baseline = json.loads(Path('claims.json').read_text(encoding='utf-8'))
llm_by_id = {row['id']: row['pred'] for row in baseline['cases']}
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, revision=REVISION).to(DEVICE).eval()
label_map = {0: 'sustentada', 1: 'inconclusiva', 2: 'contradita'}
print(model.config.id2label)


In [ ]:
rows, started = [], time.perf_counter()
for start in range(0, len(items), 16):
    batch = items[start:start + 16]
    encoded = tokenizer([x['evidence']['texto'] for x in batch], [x['claim'] for x in batch], padding=True, truncation=True, max_length=512, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        probs = model(**encoded).logits.softmax(-1).cpu()
    for item, scores in zip(batch, probs):
        pred = label_map[int(scores.argmax())]
        rows.append({'id': item['id'], 'gold': item['gold_conclusion'], 'llm': llm_by_id[item['id']], 'nli': pred, 'confidence': float(scores.max()), 'tags': item['tags']})
elapsed = time.perf_counter() - started
gold, nli, llm = [r['gold'] for r in rows], [r['nli'] for r in rows], [r['llm'] for r in rows]
labels = ['sustentada', 'contradita', 'inconclusiva']
report = {'model': MODEL, 'revision': REVISION, 'device': torch.cuda.get_device_name(0), 'n': len(rows), 'elapsed_seconds': elapsed, 'cases_per_second': len(rows)/elapsed, 'nli_accuracy': accuracy_score(gold, nli), 'nli_macro_f1': f1_score(gold, nli, labels=labels, average='macro'), 'llm_nli_agreement': accuracy_score(llm, nli), 'llm_nli_kappa': cohen_kappa_score(llm, nli, labels=labels), 'confusion_labels': labels, 'confusion': confusion_matrix(gold, nli, labels=labels).tolist(), 'cases': rows}
Path('claims_nli.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({k:v for k,v in report.items() if k != 'cases'}, ensure_ascii=False, indent=2))
files.download('claims_nli.json')
